# Domain Adaptation: MMD Loss - Hyperparameter Tuning

Goals:
- optimise the MMD weight λ
- find the optimum by grid search
- test all 3 properties

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import spearmanr
from sklearn.metrics import r2_score
import json

ROOT = r'..'
DATA_DIR = os.path.join(ROOT, 'data', 'cv_datasets')
FEAT_DIR = os.path.join(ROOT, 'features')
RES_DIR = os.path.join(ROOT, 'results', 'mmd_tuning')
os.makedirs(RES_DIR, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PROPERTIES = ['clearance', 'half_life', 'fu']
N_FOLDS = 5

# Hyperparameter Grid
MMD_WEIGHTS = [0.01, 0.05, 0.1, 0.3, 0.5, 1.0]

print("="*100)
print("Domain Adaptation: MMD Loss - Hyperparameter Tuning")
print("="*100)
print(f"Device: {DEVICE}")
print(f"Properties: {PROPERTIES}")
print(f"N Folds: {N_FOLDS}")
print(f"MMD Weights to test: {MMD_WEIGHTS}\n")

# Shared training utilities: inner train/val split + val-based early stopping (test used only for final evaluation)
sys.path.insert(0, os.path.join(ROOT, '01_model'))
from train_utils import split_train_val, fit_early_stopping, target_smiles


Domain Adaptation: MMD Loss - Hyperparameter Tuning
Device: cpu
Properties: ['clearance', 'half_life', 'fu']
N Folds: 5
MMD Weights to test: [0.01, 0.05, 0.1, 0.3, 0.5, 1.0]



In [2]:
print("Loading Graphormer embeddings...\n")

emb_path = os.path.join(FEAT_DIR, 'graphormer_cv_embeddings.npz')
emb_data = np.load(emb_path, allow_pickle=True)
emb_keys = emb_data['keys']
emb_X = emb_data['X']

smiles_to_emb = {smi: emb_X[i] for i, smi in enumerate(emb_keys)}

print(f"✓ Loaded {len(smiles_to_emb)} embeddings")
print(f"  Dim: {emb_X.shape[1]}\n")

Loading Graphormer embeddings...

✓ Loaded 58304 embeddings
  Dim: 768



In [3]:
def gaussian_kernel(x, y, sigma=1.0):
    x = x.unsqueeze(1)
    y = y.unsqueeze(0)
    distances = torch.sum((x - y) ** 2, dim=2)
    return torch.exp(-distances / (2 * sigma ** 2))

def compute_mmd(source_features, target_features, sigma=1.0):
    K_ss = gaussian_kernel(source_features, source_features, sigma)
    K_tt = gaussian_kernel(target_features, target_features, sigma)
    K_st = gaussian_kernel(source_features, target_features, sigma)
    
    n_s = source_features.shape[0]
    n_t = target_features.shape[0]
    
    mmd_loss = (1 / (n_s ** 2)) * K_ss.sum() + (1 / (n_t ** 2)) * K_tt.sum() - 2 * (1 / (n_s * n_t)) * K_st.sum()
    return torch.sqrt(torch.clamp(mmd_loss, min=1e-8))

print("✓ MMD loss defined")

✓ MMD loss defined


In [4]:
class SharedEncoder(nn.Module):
    def __init__(self, d_in=768, d_hidden=256, dropout=0.2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, d_hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.d_out = d_hidden // 2

    def forward(self, x):
        return self.encoder(x)

class TaskHead(nn.Module):
    def __init__(self, d_in, d_hidden=128):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.ReLU(),
            nn.Linear(d_hidden, 1),
        )

    def forward(self, x):
        return self.head(x).squeeze(-1)

class DomainAdaptationModel(nn.Module):
    def __init__(self, d_in=768):
        super().__init__()
        self.encoder = SharedEncoder(d_in)
        self.task_head = TaskHead(self.encoder.d_out)

    def forward(self, x):
        features = self.encoder(x)
        output = self.task_head(features)
        return output, features

print("✓ Model defined")

✓ Model defined


In [5]:
def get_smiles_col(df):
    if 'smiles' in df.columns:
        return 'smiles'
    elif 'Drug' in df.columns:
        return 'Drug'
    else:
        return None

def get_value_col(df):
    if 'value' in df.columns:
        return 'value'
    elif 'Y' in df.columns:
        return 'Y'
    else:
        return None

def load_source(prop, fold_idx):
    fold_dir = os.path.join(DATA_DIR, f'fold_{fold_idx}')
    csv_path = os.path.join(fold_dir, f'{prop}_source.csv')
    
    if not os.path.exists(csv_path):
        return None, None
    
    df = pd.read_csv(csv_path)
    smi_col = get_smiles_col(df)
    val_col = get_value_col(df)
    
    if smi_col is None or val_col is None:
        return None, None
    
    smiles = df[smi_col].astype(str).tolist()
    
    X_list = []
    y_list = []
    for smi, y in zip(smiles, df[val_col].values):
        if smi in smiles_to_emb:
            X_list.append(smiles_to_emb[smi])
            y_list.append(y)
    
    X = np.stack(X_list).astype('float32') if X_list else np.array([])
    y = np.array(y_list, dtype='float32') if y_list else np.array([])
    return X, y

def load_target(prop, fold_idx, split):
    fold_dir = os.path.join(DATA_DIR, f'fold_{fold_idx}')
    csv_path = os.path.join(fold_dir, f'{prop}_target_{split}.csv')
    
    if not os.path.exists(csv_path):
        return None, None
    
    df = pd.read_csv(csv_path)
    smi_col = get_smiles_col(df)
    val_col = get_value_col(df)
    
    if smi_col is None or val_col is None:
        return None, None
    
    smiles = df[smi_col].astype(str).tolist()
    
    X_list = []
    y_list = []
    for smi, y in zip(smiles, df[val_col].values):
        if smi in smiles_to_emb:
            X_list.append(smiles_to_emb[smi])
            y_list.append(y)
    
    X = np.stack(X_list).astype('float32') if X_list else np.array([])
    y = np.array(y_list, dtype='float32') if y_list else np.array([])
    return X, y

print("✓ Data loaders defined")

✓ Data loaders defined


In [6]:
def train_epoch_mmd(model, X_src, y_src, X_tgt, y_tgt, optimizer, criterion, mmd_w=0.1, device='cpu'):
    model.train()
    
    X_src = np.asarray(X_src, dtype='float32')
    y_src = np.asarray(y_src, dtype='float32')
    X_tgt = np.asarray(X_tgt, dtype='float32')
    y_tgt = np.asarray(y_tgt, dtype='float32')
    
    # Epochs count target batches; the source is randomly sampled from the whole set each batch (independent of source size)
    n = len(X_tgt)
    perm = np.random.permutation(n)
    
    task_loss_sum = 0.0
    mmd_loss_sum = 0.0
    n_batches = 0

    for i in range(0, n, 32):
        idx = perm[i:i + 32]
        idx_s = np.random.randint(0, len(X_src), len(idx))
        
        x_src = torch.tensor(X_src[idx_s], device=device, dtype=torch.float32)
        y_src_batch = torch.tensor(y_src[idx_s], device=device, dtype=torch.float32)
        x_tgt = torch.tensor(X_tgt[idx], device=device, dtype=torch.float32)
        y_tgt_batch = torch.tensor(y_tgt[idx], device=device, dtype=torch.float32)
        
        optimizer.zero_grad()
        
        y_pred_tgt, feat_tgt = model(x_tgt)
        task_loss = criterion(y_pred_tgt, y_tgt_batch)
        
        _, feat_src = model(x_src)
        mmd_loss = compute_mmd(feat_src, feat_tgt, sigma=1.0)
        
        total_loss = task_loss + mmd_w * mmd_loss
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        
        task_loss_sum += task_loss.item()
        mmd_loss_sum += mmd_loss.item()
        n_batches += 1

    return task_loss_sum / max(n_batches, 1), mmd_loss_sum / max(n_batches, 1)

@torch.no_grad()
def evaluate(model, X, y, device):
    model.eval()
    X = np.asarray(X, dtype='float32')
    y = np.asarray(y, dtype='float32')
    
    x = torch.tensor(X, device=device, dtype=torch.float32)
    y_pred, _ = model(x)
    y_pred = y_pred.cpu().numpy()
    
    r = np.corrcoef(y, y_pred)[0, 1]
    rho, _ = spearmanr(y, y_pred)
    r2 = r2_score(y, y_pred)
    rmse = np.sqrt(np.mean((y - y_pred) ** 2))
    mae = np.mean(np.abs(y - y_pred))
    mbe = np.mean(y_pred - y)
    
    return {'r': r, 'rho': rho, 'r2': r2, 'rmse': rmse, 'mae': mae, 'mbe': mbe}

print("✓ Training functions defined")

✓ Training functions defined


In [7]:
# GRID SEARCH: Test all MMD weights
print("\n" + "="*100)
print("HYPERPARAMETER TUNING: MMD Weight Grid Search")
print("="*100 + "\n")

tuning_results = {}  # {prop: {mmd_w: results_dict}}

for prop in PROPERTIES:
    print(f"\n{'='*80}")
    print(f"Property: {prop.upper()}")
    print(f"{'='*80}")
    
    tuning_results[prop] = {}
    
    for mmd_w in MMD_WEIGHTS:
        print(f"\n  Testing MMD Weight = {mmd_w}")
        fold_results = []
        
        for fold_idx in range(N_FOLDS):
            X_src, y_src = load_source(prop, fold_idx)
            X_tgt_train, y_tgt_train = load_target(prop, fold_idx, 'train')
            X_tgt_test, y_tgt_test = load_target(prop, fold_idx, 'test')
            
            if X_src is None or len(X_src) == 0:
                continue
            
            # The source is not truncated; everything is used (randomly sampled each batch during training)
            X_src = np.asarray(X_src, dtype='float32')
            y_src = np.asarray(y_src, dtype='float32')
            # target train → scaffold-based inner train/val (early stopping and λ selection on val, test used only for final evaluation)
            X_tgt_train, y_tgt_train, X_tgt_val, y_tgt_val = split_train_val(
                np.asarray(X_tgt_train, dtype='float32'), np.asarray(y_tgt_train, dtype='float32'), fold_idx,
                smiles=target_smiles(DATA_DIR, prop, fold_idx, 'train', smiles_to_emb))
            X_tgt_test = np.asarray(X_tgt_test, dtype='float32')
            y_tgt_test = np.asarray(y_tgt_test, dtype='float32')
            
            torch.manual_seed(fold_idx)
            np.random.seed(fold_idx)
            
            model = DomainAdaptationModel().to(DEVICE)
            optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
            criterion = nn.HuberLoss(delta=1.0)
            
            def run_epoch():
                train_epoch_mmd(model, X_src, y_src, X_tgt_train, y_tgt_train, optimizer, criterion, mmd_w, DEVICE)
            
            best_epoch, _ = fit_early_stopping(
                model, run_epoch, lambda m, X, y: evaluate(m, X, y, DEVICE), X_tgt_val, y_tgt_val)
            val_metrics = evaluate(model, X_tgt_val, y_tgt_val, DEVICE)
            metrics = evaluate(model, X_tgt_test, y_tgt_test, DEVICE)
            metrics.update({f'val_{k}': v for k, v in val_metrics.items()})
            metrics['best_epoch'] = best_epoch
            fold_results.append(metrics)
        
        if fold_results:
            results_df = pd.DataFrame(fold_results)
            tuning_results[prop][mmd_w] = {
                'rho_mean': results_df['rho'].mean(),
                'rho_std': results_df['rho'].std(),
                'r2_mean': results_df['r2'].mean(),
                'r2_std': results_df['r2'].std(),
                'rmse_mean': results_df['rmse'].mean(),
                'rmse_std': results_df['rmse'].std(),
                'val_rho_mean': float(results_df['val_rho'].mean()),
                'val_r2_mean': float(results_df['val_r2'].mean()),
            }
            
            print(f"    ρ: {results_df['rho'].mean():.4f}±{results_df['rho'].std():.4f} | R²: {results_df['r2'].mean():.4f}±{results_df['r2'].std():.4f}")
            
            # Save fold results
            results_df.to_csv(os.path.join(RES_DIR, f'{prop}_mmd{mmd_w}.csv'), index=False)


HYPERPARAMETER TUNING: MMD Weight Grid Search


Property: CLEARANCE

  Testing MMD Weight = 0.01
    ρ: 0.3477±0.0606 | R²: 0.0836±0.0496

  Testing MMD Weight = 0.05
    ρ: 0.3697±0.0542 | R²: 0.0882±0.0618

  Testing MMD Weight = 0.1
    ρ: 0.3688±0.0424 | R²: 0.1027±0.0398

  Testing MMD Weight = 0.3
    ρ: 0.3561±0.0409 | R²: 0.0946±0.0412

  Testing MMD Weight = 0.5
    ρ: 0.3583±0.0718 | R²: 0.0772±0.0864

  Testing MMD Weight = 1.0
    ρ: 0.3763±0.0338 | R²: 0.0908±0.0509

Property: HALF_LIFE

  Testing MMD Weight = 0.01
    ρ: 0.4010±0.0537 | R²: 0.0835±0.1055

  Testing MMD Weight = 0.05
    ρ: 0.4132±0.0673 | R²: 0.1086±0.0990

  Testing MMD Weight = 0.1
    ρ: 0.4035±0.0575 | R²: 0.1015±0.1061

  Testing MMD Weight = 0.3
    ρ: 0.3942±0.0573 | R²: 0.0770±0.1127

  Testing MMD Weight = 0.5
    ρ: 0.3918±0.0578 | R²: 0.0796±0.1202

  Testing MMD Weight = 1.0
    ρ: 0.3866±0.0631 | R²: 0.0691±0.1435

Property: FU

  Testing MMD Weight = 0.01
    ρ: 0.5613±0.0856 | R²: 0.2192±0

In [8]:
# Summary Report
print(f"\n\n{'='*100}")
print("TUNING RESULTS SUMMARY")
print(f"{'='*100}\n")

for prop in PROPERTIES:
    print(f"\n{prop.upper()}:")
    print(f"  {'MMD_W':<10} {'Spearman ρ':<20} {'R²':<20} {'RMSE':<20}")
    print(f"  {'-'*70}")
    
    best_rho = -1
    best_r2 = -1
    best_w_rho = None
    best_w_r2 = None
    
    for mmd_w in MMD_WEIGHTS:
        if mmd_w in tuning_results[prop]:
            res = tuning_results[prop][mmd_w]
            rho_str = f"{res['rho_mean']:.4f}±{res['rho_std']:.4f}"
            r2_str = f"{res['r2_mean']:.4f}±{res['r2_std']:.4f}"
            rmse_str = f"{res['rmse_mean']:.4f}±{res['rmse_std']:.4f}"
            
            marker = ""
            if res['val_rho_mean'] > best_rho:
                best_rho = res['val_rho_mean']
                best_w_rho = mmd_w
                marker += "(Best val ρ)"
            if res['val_r2_mean'] > best_r2:
                best_r2 = res['val_r2_mean']
                best_w_r2 = mmd_w
                marker += "(Best val R²)"
            
            print(f"  {mmd_w:<10.2f} {rho_str:<20} {r2_str:<20} {rmse_str:<20} {marker}")
    
    print(f"\n  Best MMD_W for ρ: {best_w_rho} ({best_rho:.4f})")
    print(f"  Best MMD_W for R²: {best_w_r2} ({best_r2:.4f})")



TUNING RESULTS SUMMARY


CLEARANCE:
  MMD_W      Spearman ρ           R²                   RMSE                
  ----------------------------------------------------------------------
  0.01       0.3477±0.0606        0.0836±0.0496        0.9551±0.1004        (Best val ρ)(Best val R²)
  0.05       0.3697±0.0542        0.0882±0.0618        0.9535±0.1143        (Best val ρ)
  0.10       0.3688±0.0424        0.1027±0.0398        0.9449±0.0953        (Best val R²)
  0.30       0.3561±0.0409        0.0946±0.0412        0.9499±0.1051        
  0.50       0.3583±0.0718        0.0772±0.0864        0.9582±0.1095        
  1.00       0.3763±0.0338        0.0908±0.0509        0.9513±0.1008        

  Best MMD_W for ρ: 0.05 (0.4115)
  Best MMD_W for R²: 0.1 (0.1677)

HALF_LIFE:
  MMD_W      Spearman ρ           R²                   RMSE                
  ----------------------------------------------------------------------
  0.01       0.4010±0.0537        0.0835±0.1055        0.9491±0.0745   

In [9]:
# Save tuning results
tuning_summary = {}
for prop in PROPERTIES:
    tuning_summary[prop] = {}
    for mmd_w, results in tuning_results[prop].items():
        tuning_summary[prop][str(mmd_w)] = {
            'rho_mean': float(results['rho_mean']),
            'rho_std': float(results['rho_std']),
            'r2_mean': float(results['r2_mean']),
            'r2_std': float(results['r2_std']),
            'rmse_mean': float(results['rmse_mean']),
            'rmse_std': float(results['rmse_std']),
            'val_rho_mean': float(results['val_rho_mean']),
            'val_r2_mean': float(results['val_r2_mean']),
        }

with open(os.path.join(RES_DIR, 'tuning_results.json'), 'w') as f:
    json.dump(tuning_summary, f, indent=2)

print(f"\n\n✓ Tuning results saved to {RES_DIR}")
print("\nMMD Hyperparameter Tuning Complete!")



✓ Tuning results saved to ..\results\mmd_tuning

MMD Hyperparameter Tuning Complete!
